# Lab 6: Resampling & Model Selection

The goal of this lab is to give you the tools to:

 - Perform bootstrapping: Repeatedly sample with replacement from the observed data to estimate how much a statistic, such as a regression coefficient, would vary from sample to sample.
 - Use Leave One Out Cross-Validation (LOOCV): Repeatedly fit a model while holding out one observation at a time, then use the held-out predictions to estimate prediction error.
 - Perform feature selection with:
    - Best Subset Selection
    - Forward Stepwise Selection
    - Backward Stepwise Selection

In [4]:
# Our well-trusted libraries
import os
import numpy as np
import pandas as pd

# Sci-kit learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression,LinearRegression, Lasso
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, cross_validate

# Statsmodels
import statsmodels.api as sm

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Iterator building blocks
# example: combinations('ABCD', 2) --> AB AC AD BC BD CD
from itertools import combinations

# Many were concerned with warnings. The short answer is that FutureWarning (most common)
# appears when a functionality is deprecated and will be replaced. Here's how to ignore them
# even if you should find a way to resolve them in a production environment.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

Throughout this lab, we will use an extremely simple dataset as this helps with the intuition. In essence, we will try to determine whether a person will default (i.e., fail to make their interest or principal payments on a debt) with their account balance, income, and a student indicator. More precisely:

 - `default` (str, binary): Whether the individual defaulted
 - `student` (str, binary): Whether the individual is a student
 - `balance` (float, continuous): The individual's account balance
 - `income` (float, continuous): The individual's annual income

In [6]:
PATH = (r"./data")
df_default = pd.read_csv(os.path.join(PATH, 'default.csv'), index_col=0)

display(df_default.head())
print(df_default.shape, '\n')
df_default.value_counts('default')

,default,student,balance,income
1,No,No,729.526495,44361.625074
2,No,Yes,817.180407,12106.134700
3,No,No,1073.549164,31767.138947
4,No,No,529.250605,35704.493935
5,No,No,785.655883,38463.495879


(10000, 4) 



default
No     9667
Yes     333
Name: count, dtype: int64

Let's turn the `default` and `student` into binary variables. We're only using best practices after all!

In [7]:
encoding_dict = {'Yes': 1,'No': 0}
for col in ['default', 'student']:
    df_default[col] = df_default[col].map(encoding_dict)

df_default.head()

,default,student,balance,income
1,0,0,729.526495,44361.625074
2,0,1,817.180407,12106.134700
3,0,0,1073.549164,31767.138947
4,0,0,529.250605,35704.493935
5,0,0,785.655883,38463.495879


Before we go into resampling methods, let's estimate the model with the original sample (i.e, `df_default`) to get a sense of the estimates of the coefficients as well as the standard errors associated with them.

In [8]:
X = df_default[['balance', 'income']]
X = sm.add_constant(X)

y = df_default['default']

display(X.head(), y.head())

results = sm.Logit(y, X).fit()
print(results.summary())

,const,balance,income
1,1.0,729.526495,44361.625074
2,1.0,817.180407,12106.134700
3,1.0,1073.549164,31767.138947
4,1.0,529.250605,35704.493935
5,1.0,785.655883,38463.495879


1    0
2    0
3    0
4    0
5    0
Name: default, dtype: int64

Optimization terminated successfully.
         Current function value: 0.078948
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9997
Method:                           MLE   Df Model:                            2
Date:                Wed, 04 Nov 2026   Pseudo R-squ.:                  0.4594
Time:                        16:53:45   Log-Likelihood:                -789.48
converged:                       True   LL-Null:                       -1460.3
Covariance Type:            nonrobust   LLR p-value:                4.541e-292
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.000     -12.393     -10.688
balance        0.0056      0

Here, we are interested in each coefficient estimate and its standard error. The estimated intercept is −11.5405 (SE: 0.4348), the balance coefficient is 0.005647 (SE: 0.000227), and the income coefficient is 0.00002081 (SE: 0.000004985). We will use these results as a reference when estimating coefficient uncertainty through bootstrapping

## Bootstrapping

Bootstrapping is a method for estimating sampling distributions of an estimator by resampling with replacement from the original sample. It is a flexible way to quantify uncertainty associated with an estimator or a model. Bootstrapping is especially useful because it allows us to mimic the process of selecting many observations from the population.

We use bootstrapping for estimating the distribution of a statistic (e.g. mean, variance) without using normality assumptions; in this lab, we are going to estimate hard to calculate standard errors (or confidence intervals). It is also used when the population is too small to generate a large sample.

The intuition behind the bootstrap is that we approximate the real distribution in the population by the original sample. This of course relies on the distribution in the original sample to approximate the population (which increases with n). Keep in mind: “Bootstrap confidence intervals are neither exact nor optimal, but aim instead for a wide applicability, combined with near-exact accuracy” (Efron and Hastie).

Throughout, we will use the following terminology when referring to the samples:

- **Original Sample:** The one sample of size `n` we have from the population
- **Subsample:** We draw `B` subsamples (of size `n`) __with replacement__ from the original sample. Choosing how many bootstrap replications to perform is subjective; infinity is ideal, but we will settle for “as many as possible”.

The bootstrap algorithm goes as follows:

1. Obtain the original sample from the population (size `n`)
2. Draw a subsample (with replacerment) from the original sample (size `n`)
3. Compute the estimate ($ \hat{\alpha}_{b}^{*} $) from the subsample
4. Repeat steps 2 and 3 `B` times
5. Compute $ SE_{B}(\hat{\alpha}) = \sqrt{\frac{1}{B-1} \sum_{b=1}^{B} (\hat{\alpha}_{b}^{*} - \bar{\alpha}_{b}^{*})^{2}} $ where $ \bar{\alpha}_{b}^{*} = \frac{\sum_{b=1}^{B} \hat{\alpha}_{b}^{*}}{B}$

![Bootstrap Algorithm](./bootstrap_algorithm.png)


Anyways, let's get into the code to replicate the bootstrap algorithm we just looked at. 

Step 1 is already done as `df_default` is our original sample of length `n = 10000`. 

Step 2 is to draw a subsample of `n` observations (with replacement) from the original sample. To do this, we will create the function `get_indices` to randomly select `n` indicies with replacement from our df. 

In [16]:
def get_indices(df_data, n, seed):
    '''
    Generates a random subsample (i.e., its indices)
    with replacement consisting of n observations each.
    Inputs:
        - df_data (pd.Dataframe): data to sample from
        - n (int): number indices to be randomly sampled
        - seed (int): seed to set in the random number grenerator
    Output:
        - indices (np.ndarray): array of indices forming
         the samples
    '''

    assert type(df_data) == pd.DataFrame
    assert type(n) == int, 'n must be an integer'
    
    #indices = np.random.choice(...)    # Old way
    rng = np.random.default_rng(seed)    # Creates a random number generator (rng) object 

                                         # w/ the given seed
    indices = rng.choice(df_data.index,  # Use the dataset's indices as the input
                         int(n),         # Number of indices drawn per sample
                         replace=True    # Draw samples with replacement
                        )
    return indices


Let's see if this worked the way we want:  every time we call the function, we should get a different set of indices.

In [17]:
# Example of making calls to the get_indices() function
example_indices = get_indices(df_default, len(df_default), 45)
print(example_indices[0:10])
print(get_indices(df_default, len(df_default), 45)[0:10])
print(get_indices(df_default, len(df_default), 45)[0:10])


[9246 5732 7317 5285 5319 7637 7061 8117 5627 5103]
[9246 5732 7317 5285 5319 7637 7061 8117 5627 5103]
[9246 5732 7317 5285 5319 7637 7061 8117 5627 5103]


Well that's no good!  Since we recreate the random number generator (RNG) object every time we call the get_indices() function, we always draw the same indices.  That's not what we want!

Instead, let's try creating the RNG object once, then passing that into our get_indices() function.

In [18]:
def get_indices(df_data, n, rng):
    '''
    Generates a random subsample (i.e., its indices)
    with replacement consisting of n observations each.
    Inputs:
        - df_data (pd.Dataframe): data to sample from
        - n (int): number indices to be randomly sampled
        - rng: random number grenerator (RNG) object (based on a given seed)
    Output:
        - indices (np.ndarray): array of indices forming
         the samples
    '''

    assert type(df_data) == pd.DataFrame
    assert type(n) == int, 'n must be an integer'

    indices = rng.choice(df_data.index,  # Use the dataset's indices as the input
                         int(n),         # Number of indices drawn per sample
                         replace=True    # Draw samples with replacement
                        )
    return indices

Now let's see what happens when we call the get_indices() function multiple times.  Be sure to set a random seed when you create the RNG object; this ensures the data is randomized the same way on repeated runs of your code. 

In [19]:
# Example of making calls to the new get_indices() function
seed=45
rng = np.random.default_rng(seed)    # Creates a random number generator (rng) object 
                                     # w/ the given seed
print(get_indices(df_default, len(df_default), rng)[0:10])
print(get_indices(df_default, len(df_default), rng)[0:10])
print(get_indices(df_default, len(df_default), rng)[0:10])


[9246 5732 7317 5285 5319 7637 7061 8117 5627 5103]
[7977 8221 9651  923 8376 7360 4436 3309 1829 8489]
[ 775 5552 9983 2953 9520 4226 4153 4786 8916  894]


That's what we wanted!!!  Note that if you re-run the cell, you'll get the same sequence of random indices every time.  That's crucial for replication.    


Also, remember, there is replacement, meaning that an observation can occur 0, 1, or multiple times in the subsample. Here's proof that it works:

In [20]:
rng = np.random.default_rng(45)
np.asarray(np.unique(get_indices(df_default, 10000, rng), return_counts=True)).T[-10:]

array([[ 9986,     1],
       [ 9987,     1],
       [ 9989,     3],
       [ 9991,     2],
       [ 9993,     1],
       [ 9995,     2],
       [ 9996,     2],
       [ 9997,     2],
       [ 9998,     1],
       [10000,     1]])

Step 3 is to estimate the coefficients with each subsample. We create a function `boot_fn` which performs the `logit` on each subsample in order to get the estimates.

In [21]:
# Similar to boot.fn in the exercise
def boot_fn(df_data, index, constant=True, features=['balance','income'], target='default'):
    '''
    Runs a logistic regression (with a constant) on only the specified
    indices within the data (i.e., on a single subsample).

    Inputs:
        - df_data (pd.Dataframe): data to sample from
        - indices (np.ndarray): array of indices forming the samples
        - features (lst of str): the name of the features
        - target (str): the name of the target

    Output:
        - coefficients (lst of float): the coefficients
    '''
    X = df_data[features].loc[index]
    if constant:
        X = sm.add_constant(X)
    y = df_data[target].loc[index]
    
    lr = sm.Logit(y,X).fit(disp=0)
    coefficients = lr.params.tolist()

    return coefficients

In [22]:
# Example of making a call to the boot_fn function
intercept, coef_balance, coef_income = boot_fn(df_default, get_indices(df_default, 10000, rng))
print(f'Coefficients of a single subsample:\n\tIntercept:\t{round(intercept, 2)}\n\tBalance:\t{round(coef_balance, 2)}\n\tIncome:\t\t{round(coef_income, 2)}')

Coefficients of a single subsample:
	Intercept:	-11.77
	Balance:	0.01
	Income:		0.0


In step 4, we want to repeat steps 2 and 3 `B` number of times to create many subsamples and estimate their coefficients.  
In step 5, we want to compute the mean and standard deviation based on an average of all the estimates.   
We will do both in the `boot` function.

# How long will this take to run?

Before running the full bootstrap, let’s practice measuring how long an operation takes. The next cell uses a five-second pause to demonstrate elapsed-time measurement; we will then apply the same approach to our bootstrap procedure. Timing code is useful in practice because bootstrapping, cross-validation, and model selection require repeated model fits. Understanding their computational cost helps us plan resources, compare approaches, and decide which analyses are feasible before starting a lengthy run.

In [23]:
import time

start_time = time.time()
# I want to take a 5 second nap
time.sleep(5)

end_time = time.time()

print(end_time - start_time)


5.000423192977905


In [26]:
def boot(df_data, func, B, rng):
    '''
    Executing a bootstrap over B subsamples to estimate the (mean of) coefficients
    and their associated standard errors.

    Inputs:
        - df_data (pd.Dataframe): data to sample from
        - func (fn): function executing the regression
        - B (int): number of subsamples
        - rng: random number grenerator (RNG) object (based on a given seed)
    
    Ouput:
        - restults (dict of dicts): bootstrapped coefficients & associated standard errors
    '''
    
    # Step 4
    coef_intercept = []
    coef_balance = []
    coef_income = []

    coefs = ['intercept', 'balance', 'income']
    output = {coef: [] for coef in coefs}
    for b in range(B):
        reg_out = func(df_data, get_indices(df_data, len(df_data), rng))
        for i, coef in enumerate(coefs):
            output[coef].append(reg_out[i])

    # Step 5
    results = {}
    for coef in coefs:
        results[coef] = {
            'estimate': np.mean(output[coef]),
            'std_err': np.std(output[coef], ddof=1)
        }
    
    return results

Essentially, the three functions we created are the building blocks to performing the bootstrap algorithm. 

Now, we can finally run our own bootstrap on the data we have! 

In [27]:
start_time = time.time()

rng = np.random.default_rng(45)
results = boot(df_default, boot_fn, 1000, rng)
for i, x in results.items():
    print(f"{i.capitalize()}:\n\tEstimate: {x['estimate']}\n\tStandard Error: {x['std_err']}")

end_time = time.time()

print(end_time - start_time)


Intercept:
	Estimate: -11.579845617304015
	Standard Error: 0.43681305179132496
Balance:
	Estimate: 0.005669135992073182
	Standard Error: 0.00023289247032756436
Income:
	Estimate: 2.0811359498949125e-05
	Standard Error: 4.604572739765029e-06
7.135392189025879


**The average coefficient estimates across the bootstrap samples are similar to those from the original sample. Bootstrapping gives us a sense of how much our coefficient estimates would vary from sample to sample. We estimate each coefficient’s standard error using the standard deviation of its estimates across the bootstrap samples.**

#### Side Bar:

In [28]:
# the help() function will return the docstring of your function; 
# you can remind yourself what a function does at any point throughout your code
help(get_indices)

Help on function get_indices in module __main__:

get_indices(df_data, n, rng)
    Generates a random subsample (i.e., its indices)
    with replacement consisting of n observations each.
    Inputs:
        - df_data (pd.Dataframe): data to sample from
        - n (int): number indices to be randomly sampled
        - rng: random number grenerator (RNG) object (based on a given seed)
    Output:
        - indices (np.ndarray): array of indices forming
         the samples



We have now used bootstraps to estimate how much our regression coefficients vary across samples. The standard deviation of each coefficient across these replications provides a bootstrap estimate of its standard error. For example, the bootstrap standard error for the balance coefficient is approximately 0.000233, compared with 0.000227 from the original regression. This illustrates how resampling can estimate coefficient uncertainty numerically.

# Leave One Out Cross-Validation


## Note that we could also use log loss instead of the Brier score (which in a binary outcome scenario is equivalent to MSE). 

We already went over cross-validation in previous labs. However, we have yet to cover its most extreme version; Leave One Out Cross Validation (LOOCV).

As its name suggests, LOOCV is simply cross validation where the test set contains a single observation. There are a few pros and cons as to why you would choose to use cross-validation:

- **Pros:**
    - Low Upward Bias: because you only lose one observation when fitting the model on the training data
    - Final Test Error Not Variable: The test error will be the same every time you run LOOCV on the dataset because it goes through the same validation process every time. 
- **Cons:**
    - Computationally Costly: Requires estimating the model `n` different times
    - Prone to higher variance (especially when n is small)

In [29]:
squared_errors = []

#How will the random seed affect LOOCV?
# np.random.default_rng(5)
np.random.default_rng(1)
# np.random.default_rng(55)

# Iterate over the entire dataset one observation at a time
print('Iteration Milestones:...')
for i in df_default.index:
    # All except observation i is our training set
    train = df_default.iloc[df_default.index != i,:]
    # Only observation i is our test set
    test = df_default.iloc[df_default.index == i,:]

    # Fit the model and gather the squared error
    ols = LinearRegression().fit(train[['balance', 'income']], train['default'])
    test_predicted = ols.predict(test[['balance', 'income']])
    test_actual = test[['default']]
    squared_error = np.power(test_predicted - test_actual, 2)

    # Store the squared error
    squared_errors.append(squared_error.values[0][0])

    if i % 1000 == 0:
        print(i)

print(f'\n{squared_errors[:10]}\n')

# From the squared errors, get the Mean Squared Error (MSE)
print(f'MSE using LOOCV: {round(np.mean(squared_errors), 5)}')

Iternation Milestones:...
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000

[np.float64(0.000592738082377867), np.float64(0.00044313658761178823), np.float64(0.004082543662994613), np.float64(3.650933906998175e-05), np.float64(0.0008426104370810316), np.float64(0.0010518241573572369), np.float64(0.0007861782470870658), np.float64(0.0005042958841561064), np.float64(0.006093171030518951), np.float64(0.006208461720247741)]

MSE using LOOCV: 0.02824


Notice how certain errors are way larger than others? Those are outliers, but they average out over the size of dataset so we don't need to give them much attention.

We can compare our results from LOOCV to regular KFold CV. An alternative way of doing k-fold cross validation is with the `cross_val_score` that is also in the sci-kit learn library. Here is how to use it with shuffling/randomizing of the data.

In [35]:
from sklearn.model_selection import KFold

X = df_default[['balance','income']]
Y = df_default['default']
ols = LinearRegression()

cv = KFold(5,              # Number of folds
           shuffle=True,   # Randomizes observations into folds if true
           random_state=9  # Affects how randomization occurs
          )  

cv_scores = cross_val_score(ols,      # Specified model
                            X,        # Features
                            Y,        # Target
                            cv=cv,    # Number of folds
                            scoring='neg_mean_squared_error'  # Metric
                           )

print(cv_scores)

print(np.mean(-cv_scores))

[-0.02271776 -0.03038288 -0.02430095 -0.03140294 -0.0325379 ]
0.02826848555621416


What if we tried to perform LOOCV on a smaller dataset and compare to the performance of KFold CV? 

To simulate this, we will randomly select smaller dataframes and perform LOOCV to compare how LOOCV changes as n increases. To do this, we can reuse our get_indices function from before.

In [36]:
#Compare the test errors from KFoldCV to LOOCV over different n size datasets
for n in [1000, 2500, 5000, 7500, 10000]:
    # Subset datasets of different sized n
    n_indices = get_indices(df_default, n, rng)
    subset = df_default.loc[n_indices].reset_index(drop=True)

    # Use the same code to get the LOOCV
    squared_errors = []
    for i in subset.index:
        # All except observation i is our training set
        train = subset.iloc[subset.index != i,:]
        # Only observation i is our test set
        test = subset.iloc[subset.index == i,:]

        # Fit the model and gather the squared error
        ols = LinearRegression().fit(train[['balance', 'income']], train['default'])
        test_predicted = ols.predict(test[['balance', 'income']])
        test_actual = test[['default']]
        squared_error = np.power(test_predicted - test_actual, 2)

        # Store the squared error
        squared_errors.append(squared_error.values[0][0])  
    print('MSE using LOOCV using n = '+ str(n) + f': {round(np.mean(squared_errors), 5)}')


    # Use the same code to get the error from the KFold CV
    X = subset[['balance','income']]
    Y = subset['default']
    ols = LinearRegression()

    cv = KFold(5, shuffle=True, random_state=9)  

    cv_scores = cross_val_score(ols, X, Y, cv = cv, scoring = 'neg_mean_squared_error')

    print('MSE using KFold CV using n = '+ str(n) + f': {round(np.mean(-cv_scores), 5)}')
    

MSE using LOOCV using n = 1000: 0.0275
MSE using KFold CV using n = 1000: 0.02751
MSE using LOOCV using n = 2500: 0.03003
MSE using KFold CV using n = 2500: 0.03
MSE using LOOCV using n = 5000: 0.0267
MSE using KFold CV using n = 5000: 0.02677
MSE using LOOCV using n = 7500: 0.03071
MSE using KFold CV using n = 7500: 0.03072
MSE using LOOCV using n = 10000: 0.02987
MSE using KFold CV using n = 10000: 0.02992


From our mini experiment, we see that the KFold CV performs almost equally to LOOCV when applied to the `df_default` dataset. In general, we prefer KFold because it takes less time to run computationally.

## Best Subset Selection


## mlextend's Exhaustive Feature Selection

In [ ]:
from sklearn.linear_model import LinearRegression
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS

X = df_default[['balance', 'income', 'student']]
y = df_default['default']

model = LinearRegression()

efs = EFS(
    model,
    min_features=1,
    max_features=3,  
    scoring='neg_root_mean_squared_error',  
    cv=5,  
    n_jobs=-1 
)

efs.fit(X, y)

# Method 1:
best_subset = max(efs.subsets_.values(), key=lambda x: x['avg_score']) 
best_features = best_subset['feature_names']  
best_rmse = -1*best_subset['avg_score']  
print(f"Best Model: Features = {best_features}, RMSE = {round(best_rmse, 5)}")

# Method 2: 
best_subset = list(efs.best_feature_names_)
best_rmse = -1*efs.best_score_
print(f"Best Model: Features = {best_features}, RMSE = {round(best_rmse, 5)}")


Features: 7/7

Best Model: Features = ('balance', 'student'), RMSE = 0.16796
Best Model: Features = ('balance', 'student'), RMSE = 0.16796


We can see that the combination of features that best predict whether an individual will default is `student` and `balance`. This hints that `income` is either an irrelevant variable when the others are given or `income` is highly correlated with at least one of the other variables and thus only adds noise when predicting on the test set.

Anyways, now we know which features form the best subset. However, computing all possible combinations is not always feasible. As seen in class, the more features, the larger the data, and the more computationally intensive the model, the longer it takes to compute all possible combinations. Therefore, there is a need for other feature selections methods.

# sklearn's Sequential Feature Selector, First forwards

In [153]:
###################################
## Using sklearn's Sequential Feature Selector:  General Setup

df_default['CONSTANT'] = 1 

X = df_default[['balance', 'income', 'student']]
y = df_default['default']


ols = LinearRegression()

rs = 314

kfcv = KFold(5,               
             shuffle=True,    
             random_state=rs  
             )

### Forward Stepwise Selection

The idea of this model is that we want to add the variable that provides the biggest improvement to our model (the one that yields the smallest error). We start with a model that has a constant as its only feature and continue until we run out of variables or see an increase in our error rate.

Its algorithm is fairly intuitive:

1. For k = 0, let $ M_{0} $ denote the null model (no Xs)
2. For k = 1, ..., p:
    - Consider all p - k + 1 models that add one predictor to $ M_{k-1} $
    - Pick the best (smallest RSS or largest $ R^{2} $) of these models and call it $ M_{k} $
3. Select the single best (CV test error, $ C_{p} $, AIC, BIC, or adjusted $ R^{2} $) model among $ M_{0}$, ..., $ M_{p}$. For the purposes of this lab, we'll use the RSME. You can try other metrics as well!

In [154]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error, make_scorer

scoring = "neg_root_mean_squared_error" # RMSE scoring

sfs_forward = SequentialFeatureSelector(ols,
                                        n_features_to_select="auto",
                                        direction="forward",
                                        scoring=scoring,  # Use RMSE instead of R²
                                        cv=kfcv)

# Fit the feature selector
sfs_forward.fit(X, y)

# Extract selected features
selected_features = X.columns[sfs_forward.get_support()].tolist()

print("\nBest Forward Stepwise Model by RMSE:")
print(selected_features)

best_model_rmse = -cross_val_score(ols, X[selected_features], y, cv=kfcv, scoring=scoring).mean()
print(f"\nBest Model RMSE: {round(best_model_rmse, 5)}")


Best Forward Stepwise Model by RMSE:
['balance']

Best Model RMSE: 0.16798


We see that forward stepwise selection chooses the model with `balance`. This disagrees with the "Best" subset selection.  Note there is no guarantee that we do find the best model from all possible combinations with forward stepwise selection.

### Backward Stepwise Selection
The idea of this model is that we want to remove the variable that is not adding predictive power to our model (one that we can remove without increasing our error). We start with a full model (using all of our variables) and drop variables until we see an increase in our error rate or end up with only a constant. The idea remains the exact same as in forward stepwise selection, but we start at a different point.

The algorithm goes as follows:

1. For k = p, let $ M_{p} $ denote the fully specified model (all Xs)
2. For k = p-1, ..., 0:
    - Consider all p - k + 1 models that remove one predictor from $ M_{k+1} $
    - Pick the best (smallest RSS or largest $ R^{2} $) of these models and call it $ M_{k} $
3. Select the single best (CV test error, $ C_{p} $, AIC, BIC, or adjusted $ R^{2} $) model among $ M_{0}$, ..., $ M_{p}$. Again, we're using RSME here. 

In [155]:
sfs_backward = SequentialFeatureSelector(ols,
                                         n_features_to_select="auto",  # Automatically selects the best subset
                                         direction="backward",
                                         scoring=scoring,
                                         cv=kfcv)

# Fit the feature selector
sfs_backward.fit(X, y)

# Extract selected features
selected_features = X.columns[sfs_backward.get_support()].tolist()

print("\nBest Backward Stepwise Model by RMSE:")
print(selected_features)

# Evaluate RMSE of the best model
best_model_rmse = -cross_val_score(ols, X[selected_features], y, cv=kfcv, scoring=scoring).mean()
print(f"\nBest Model RMSE: {round(best_model_rmse, 5)}")


Best Backward Stepwise Model by RMSE:
['balance', 'student']

Best Model RMSE: 0.1679


Exception ignored in: <function ResourceTracker.__del__ at 0x7f9f0c6b4c20>
Traceback (most recent call last):
  File "/home/cclapp/miniconda3/envs/dap/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/home/cclapp/miniconda3/envs/dap/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/home/cclapp/miniconda3/envs/dap/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7ff509030cc0>
Traceback (most recent call last):
  File "/home/cclapp/miniconda3/envs/dap/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/home/cclapp/miniconda3/envs/dap/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/home/cclapp/miniconda3/envs/dap/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child 

Under the current settings, backward selection removes income and retains balance and student, producing a mean cross-validation RMSE of approximately 0.16790. Starting with all three predictors, the procedure evaluates candidate removals using cross-validation RMSE. This illustrates how we can assess predictive performance while simplifying a model.